In [1]:
import os
import pandas as pd
import numpy as np
import mindspore as ms
from mindspore import nn, Tensor
from sklearn.preprocessing import MinMaxScaler
import moxing as mox

# Memanggil model dari file transformers.py
from transformers import LSTMTransformer

ms.set_context(mode=ms.PYNATIVE_MODE)

INFO:root:Using MoXing-v2.1.0.5d9c87c8-5d9c87c8
INFO:root:Using OBS-Python-SDK-3.20.9.1


In [2]:
obs_data_path = 'obs://mindspore-dataset-1/Data/data_bersih.csv'
local_data_path = './data_bersih.csv'
obs_model_dir = 'obs://mindspore-dataset-1/output_model2/Transformer/'

print("Download dataset dari OBS...")
mox.file.copy(obs_data_path, local_data_path)

df = pd.read_csv(local_data_path)

# ==========================================
# PROSES DATA CLEANSING (TIME-SERIES HANDLING)
# ==========================================
# 1. Cek jumlah NaN awal (untuk monitoring)
print(f"Jumlah NaN awal - Fitur: {df[['Suhu', 'Kelembaban', 'CO2', 'NH3', 'Cahaya']].isna().sum().sum()}")
print(f"Jumlah NaN awal - Target: {df[['Inline', 'Exhaust', 'Heater']].isna().sum().sum()}")

# 2. Interpolasi Linear untuk mengisi nilai yang kosong berdasarkan tren deret waktu
# limit_direction='both' memastikan jika NaN ada di baris pertama/terakhir tetap terisi
df[['Suhu', 'Kelembaban', 'CO2', 'NH3', 'Cahaya']] = df[['Suhu', 'Kelembaban', 'CO2', 'NH3', 'Cahaya']].interpolate(method='linear', limit_direction='both')
df[['Inline', 'Exhaust', 'Heater']] = df[['Inline', 'Exhaust', 'Heater']].interpolate(method='linear', limit_direction='both')

# 3. Jaga-jaga: Jika masih ada sisa NaN yang tidak bisa diinterpolasi, gunakan forward fill lalu backward fill
df = df.ffill().bfill()

# 4. Verifikasi akhir pastikan NaN sudah 0
print(f"Jumlah NaN setelah cleansing - Fitur: {df[['Suhu', 'Kelembaban', 'CO2', 'NH3', 'Cahaya']].isna().sum().sum()}")
print(f"Jumlah NaN setelah cleansing - Target: {df[['Inline', 'Exhaust', 'Heater']].isna().sum().sum()}")
print("-" * 50)

# mengambil nilai matriks numpy
features = df[['Suhu', 'Kelembaban', 'CO2', 'NH3', 'Cahaya']].values
targets = df[['Inline', 'Exhaust', 'Heater']].values

# Normalisasi Fitur (Input)
scaler_features = MinMaxScaler()
features_scaled = scaler_features.fit_transform(features)

# Normalisasi Target (Output)
scaler_targets = MinMaxScaler()
targets_scaled = scaler_targets.fit_transform(targets)

/home/ma-user/anaconda3/envs/MindSpore/lib/python3.7/site-packages/requests/__init__.py:104: RequestsDependencyWarning: urllib3 (1.26.12) or chardet (5.2.0)/charset_normalizer (2.0.12) doesn't match a supported version!
  RequestsDependencyWarning)


Download dataset dari OBS...
Jumlah NaN awal - Fitur: 160
Jumlah NaN awal - Target: 96
Jumlah NaN setelah cleansing - Fitur: 0
Jumlah NaN setelah cleansing - Target: 0
--------------------------------------------------


In [3]:
# =========================================================
# 3. PEMBUATAN SEQUENCE & DATA SPLITTING
# =========================================================
SEQ_LENGTH = 18

def create_sequences(data, target, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])
        y.append(target[i+seq_length])
    return np.array(X, np.float32), np.array(y, np.float32)

# Membuat sekuens dari data yang sudah bersih dari NaN
X, y = create_sequences(features_scaled, targets_scaled, SEQ_LENGTH)

# Menentukan titik potong (80% data awal untuk training, 20% data terakhir untuk testing)
split = int(0.8 * len(X))

# Membagi data tanpa di-shuffle (diacak) agar urutan waktu tetap terjaga
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# Menampilkan verifikasi jumlah baris hasil pembagian
print("=== VERIFIKASI PEMBAGIAN DATASET ===")
print(f"Total seluruh sampel sekuens  : {X.shape[0]} sampel")
print(f"Jumlah sampel Data Training (80% awal)  : {X_train.shape[0]} sampel")
print(f"Jumlah sampel Data Testing  (20% terakhir): {X_test.shape[0]} sampel")

=== VERIFIKASI PEMBAGIAN DATASET ===
Total seluruh sampel sekuens  : 4899 sampel
Jumlah sampel Data Training (80% awal)  : 3919 sampel
Jumlah sampel Data Testing  (20% terakhir): 980 sampel


In [4]:
# ==========================================
# ATUR HYPERPARAMETER DI SINI
# ==========================================
PARAMS = {
    'input_dim': 5,
    'lstm_units': 128,       # Coba naikkan/turunkan (misal: 32, 64, 128)
    'num_heads': 4,         # Harus habis membagi lstm_units (misal: 2, 4, 8)
    'ff_dim': 256,          # Dimensi feed forward internal
    'dropout_rate': 0.15,   # Regularisasi untuk mencegah stagnasi / overfit
    'learning_rate': 0.0001,# LR lebih kecil agar tidak loncat melompati local minima
    'epochs': 50,
    'batch_size': 32
}

# Inisialisasi model dengan parameter dinamis
model = LSTMTransformer(
    input_dim=PARAMS['input_dim'],
    lstm_units=PARAMS['lstm_units'],
    num_heads=PARAMS['num_heads'],
    ff_dim=PARAMS['ff_dim'],
    dropout_rate=PARAMS['dropout_rate']
)

loss_fn = nn.MSELoss()
optimizer = nn.Adam(model.trainable_params(), learning_rate=PARAMS['learning_rate'])

net = nn.WithLossCell(model, loss_fn)
train_step = nn.TrainOneStepCell(net, optimizer)
train_step.set_train()

def train(X, y, epochs, batch_size):
    for epoch in range(epochs):
        total_loss = 0
        total_sample = 0

        for i in range(0, len(X), batch_size):
            x_batch = Tensor(X[i:i+batch_size], ms.float32)
            y_batch = Tensor(y[i:i+batch_size], ms.float32)

            loss = train_step(x_batch, y_batch)
            total_loss += loss.asnumpy()
            total_sample += 1

        print(f"Epoch {epoch+1}, Loss: {total_loss/total_sample:.5f}")

print("\nMulai training...")
train(X_train, y_train, epochs=PARAMS['epochs'], batch_size=PARAMS['batch_size'])


Mulai training...
Epoch 1, Loss: 0.09413
Epoch 2, Loss: 0.09289
Epoch 3, Loss: 0.09350
Epoch 4, Loss: 0.09367
Epoch 5, Loss: 0.09411
Epoch 6, Loss: 0.09379
Epoch 7, Loss: 0.09370
Epoch 8, Loss: 0.09326
Epoch 9, Loss: 0.09254
Epoch 10, Loss: 0.09221
Epoch 11, Loss: 0.09176
Epoch 12, Loss: 0.09136
Epoch 13, Loss: 0.09116
Epoch 14, Loss: 0.09097
Epoch 15, Loss: 0.09066
Epoch 16, Loss: 0.09046
Epoch 17, Loss: 0.09035
Epoch 18, Loss: 0.09003
Epoch 19, Loss: 0.08997
Epoch 20, Loss: 0.08984
Epoch 21, Loss: 0.08999
Epoch 22, Loss: 0.09021
Epoch 23, Loss: 0.08972
Epoch 24, Loss: 0.08988
Epoch 25, Loss: 0.08977
Epoch 26, Loss: 0.08963
Epoch 27, Loss: 0.08985
Epoch 28, Loss: 0.08973
Epoch 29, Loss: 0.08963
Epoch 30, Loss: 0.08968
Epoch 31, Loss: 0.08976
Epoch 32, Loss: 0.08979
Epoch 33, Loss: 0.08984
Epoch 34, Loss: 0.09019
Epoch 35, Loss: 0.09024
Epoch 36, Loss: 0.09038
Epoch 37, Loss: 0.09006
Epoch 38, Loss: 0.09015
Epoch 39, Loss: 0.09011
Epoch 40, Loss: 0.08977
Epoch 41, Loss: 0.08976
Epoch 

In [7]:
# =========================================================
# 5. EVALUASI DATA TESTING (20% TERAKHIR)
# =========================================================
# Mengubah model ke mode evaluasi/testing
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
model.set_train(False)

# PERBAIKAN: Menggunakan X_test (bukan X_val)
pred_scaled = model(Tensor(X_test, ms.float32)).asnumpy()

# Mengembalikan hasil prediksi dan target test ke skala semula (Denormalisasi)
pred_original = scaler_targets.inverse_transform(pred_scaled)
y_test_original = scaler_targets.inverse_transform(y_test)

# Menghitung Metrik Evaluasi pada skala asli sensor
mae = mean_absolute_error(y_test_original, pred_original)
rmse = np.sqrt(mean_squared_error(y_test_original, pred_original))
r2 = r2_score(y_test_original, pred_original)

print("=== HASIL EVALUASI DATA TESTING (SKALA ASLI) ===")
print(f"Test MAE:  {mae:.4f}")
print(f"Test RMSE: {rmse:.4f}")
print(f"Test R2:   {r2:.4f}")

=== HASIL EVALUASI DATA TESTING (SKALA ASLI) ===
Test MAE:  16.3673
Test RMSE: 22.4088
Test R2:   -0.4488


In [8]:
local_model_path = './lstm_transformer.ckpt'

print("\nMenyimpan model...")
ms.save_checkpoint(model, local_model_path)

print("Upload ke OBS...")
mox.file.copy(local_model_path, obs_model_dir + 'lstm_transformer.ckpt')
print("Selesai upload!")


Menyimpan model...
Upload ke OBS...
Selesai upload!
